# Train YOLO26m ở 1280 để đếm mụn (ACNE04v2)

Mỗi lần mở Colab thì chạy lại S1, S2, S3 rồi chạy tiếp cell đang dở. Colab free chỉ có khoảng 5 tiếng GPU nên cell train tự dừng trước khi hết giờ (`SESSION_HOURS`), lần sau chạy lại là train tiếp. Các bước đánh giá đều lưu lên Drive, phần nào xong rồi thì bỏ qua.

| Mục | Làm gì | Ghi chú |
|---|---|---|
| S1-S3 | Cài đặt, dữ liệu, hàm tiện ích | chạy mỗi phiên |
| 0 | Thống kê dữ liệu | xem tỉ lệ mụn dưới 8 px theo imgsz |
| 1 | Kiểm tra nhãn và rò rỉ train/val/test | nên chạy trước khi train |
| 2 | Train dài yolo26m ở 1280 | 100 epoch, có early stopping |
| 3 | Overfit và độ ổn định | đánh giá các checkpoint |
| 4 | Quét conf, end2end, iou trên val | chốt cấu hình |
| 5 | Đánh giá test | có khoá chống chạy lại |
| 6 | Phân tích lỗi | ảnh sai nhiều nhất, crop để soát nhãn |
| 7 | SAHI | không bắt buộc, chỉ thử khi mục 6 cho thấy cần |
| 8 | Đóng gói cho RTX 3050 | best.pt, config, script đếm |
| 9 | Chuyển run giữa Colab và Kaggle | xem thêm `acne_kaggle_resume.ipynb` |

Lý do của từng lựa chọn nằm trong `bao_cao_acne_yolo26m.md`.


In [ ]:
# Cell S1: Cài thư viện, mount Drive, cấu hình chung
import os, json, time, pickle, hashlib, subprocess, sys, shutil, importlib.util
from pathlib import Path

# colab mỗi phiên là máy mới nên cài lại, giữ đúng bản lúc screening
ULTRA_VER = '8.4.192'
if importlib.util.find_spec('ultralytics') is None:
    print(f'Cài ultralytics=={ULTRA_VER} ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'ultralytics=={ULTRA_VER}'], check=True)

from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
import torch, ultralytics
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Đường dẫn
DRIVE_ROOT = Path('/content/drive/MyDrive/acne_yolo26')  # lưu hết vào đây cho khỏi mất
RUNS = DRIVE_ROOT / 'runs'
EVAL = DRIVE_ROOT / 'eval'  # cache dự đoán với bảng kết quả
for p in (RUNS, EVAL): p.mkdir(parents=True, exist_ok=True)
DATA_YAML = '/content/acne_data_fixed.yaml'
RAR_ON_DRIVE = '/content/drive/MyDrive/acne04v2_yolo.rar'

# Cấu hình thí nghiệm
IMGSZ = 1280  # ở 640 có 52% mụn dưới 8px, nhỏ quá
RUN_NAME = f'y26m_{IMGSZ}'
SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_DIR = {'train': 'train', 'val': 'valid', 'test': 'test'}  # val của ultralytics là thư mục valid

SESSION_T0 = time.time()
SESSION_HOURS = 4.5  # colab free đươc tầm 5 tiếng, dừng trước khi bị cắt

In [ ]:
# Cell S2: giải nén dữ liệu nếu chưa có, tạo data.yaml theo đường dẫn Linux
EXTRACT_DIR = Path('/content/acne_dataset')

def find_data_root(base):
    """Tìm thư mục có train/images và valid/images."""
    if not base.exists(): return None
    for p in [base, *sorted(base.rglob('*'))]:
        if p.is_dir() and (p / 'train' / 'images').is_dir() and (p / 'valid' / 'images').is_dir():
            return p
    return None

if find_data_root(EXTRACT_DIR) is None:
    assert Path(RAR_ON_DRIVE).exists(), f'Không thấy {RAR_ON_DRIVE}'
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    local_rar = Path('/content/acne04v2_yolo.rar')
    print('Copy .rar từ Drive về máy Colab...')
    shutil.copy(RAR_ON_DRIVE, local_rar)
    print('Giải nén...')
    if shutil.which('unrar') is None:
        subprocess.run('apt-get -qq install -y unrar > /dev/null', shell=True)
    ok = subprocess.run(['unrar', 'x', '-o+', '-inul', str(local_rar), str(EXTRACT_DIR) + '/']).returncode == 0
    if not ok:  # unrar lỗi thì thử 7z
        ok = subprocess.run(['7z', 'x', '-y', f'-o{EXTRACT_DIR}', str(local_rar)], stdout=subprocess.DEVNULL).returncode == 0
    assert ok, 'Giải nén lỗi'
    local_rar.unlink()

DATA_ROOT = find_data_root(EXTRACT_DIR)
assert DATA_ROOT is not None, f'Không tìm thấy thư mục có train/images và valid/images trong {EXTRACT_DIR}'
print('DATA_ROOT =', DATA_ROOT)
for s in ['train', 'valid', 'test']:
    d = DATA_ROOT / s
    n_img = len(list((d / 'images').glob('*'))) if (d / 'images').exists() else 0
    n_lab = len(list((d / 'labels').glob('*.txt'))) if (d / 'labels').exists() else 0
    print(f'  {s:5s}: {n_img} ảnh | {n_lab} file nhãn')

orig_yaml = list(DATA_ROOT.glob('*.yaml'))
names = yaml.safe_load(open(orig_yaml[0])).get('names', ['acne']) if orig_yaml else ['acne']
yaml.safe_dump(dict(path=str(DATA_ROOT), train='train/images', val='valid/images', test='test/images', names=names),
               open(DATA_YAML, 'w'), allow_unicode=True)
print(open(DATA_YAML).read())

In [ ]:
# Cell S3: hàm tiện ích và cấu hình train, chạy lại mỗi phiên
from PIL import Image
from ultralytics.data.utils import exif_size

# thang Hayashi, đếm trên nửa mặt. Gộp nặng với rất nặng nên còn 3 mức
SEV_BINS = [5, 20]
SEV_NAMES = ['nhẹ', 'vừa', 'nặng']
SEV_ASCII = ['nhe', 'vua', 'nang']  # cv2.putText ko vẽ đươc dấu tiếng việt
def severity(counts):
    return np.digitize(np.asarray(counts), np.array(SEV_BINS), right=True)   # <=5 là 0, 6-20 là 1, >20 là 2

def list_images(split):
    d = DATA_ROOT / SPLIT_DIR[split] / 'images'
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT)

def label_path(img):
    return Path(img).parent.parent / 'labels' / (Path(img).stem + '.txt')

def read_labels(img):
    """(N,5) cls xc yc w h chuẩn hoá; None nếu thiếu file nhãn."""
    lp = label_path(img)
    if not lp.exists(): return None
    rows = [[float(x) for x in ln.split()] for ln in lp.read_text().strip().splitlines() if len(ln.split()) == 5]
    return np.array(rows, dtype=float).reshape(-1, 5)

def gt_xyxy(img, shape_hw):
    lab = read_labels(img)
    if lab is None or len(lab) == 0: return np.zeros((0, 4))
    h, w = shape_hw[:2]
    xc, yc, bw, bh = lab[:, 1] * w, lab[:, 2] * h, lab[:, 3] * w, lab[:, 4] * h
    return np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1)

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(gt, pr, conf, iou_thr=0.5):
    """Ghép theo conf giảm dần như lúc tính AP, gconf[g] = 0 nếu nhãn g không ghép được."""
    tp = np.zeros(len(pr), bool); gconf = np.zeros(len(gt)); used = np.zeros(len(gt), bool)
    if len(gt) and len(pr):
        order = np.argsort(-conf); ious = box_iou(pr[order], gt)
        for k, j in enumerate(order):
            row = np.where(used, -1.0, ious[k]); g = int(row.argmax())
            if row[g] >= iou_thr:
                tp[j] = True; used[g] = True; gconf[g] = conf[j]
    return tp, gconf

def collect_preds(weights, split='val', imgsz=IMGSZ, end2end=True, iou=0.7, max_det=500, augment=False, batch=8):
    """Dự đoán một lần ở conf rất thấp rồi cache lại để quét ngưỡng sau."""
    w = Path(weights)
    tag = f"{w.parent.parent.name}_{w.stem}_{split}_{imgsz}_e2e{int(end2end)}_iou{iou}_md{max_det}_aug{int(augment)}"
    out = EVAL / 'preds' / f'{tag}.pkl'; out.parent.mkdir(exist_ok=True)
    if out.exists(): return pickle.load(open(out, 'rb'))
    model = YOLO(str(w)); imgs = list_images(split); recs = []
    for i in range(0, len(imgs), batch):
        chunk = [str(p) for p in imgs[i:i + batch]]
        res = model.predict(chunk, imgsz=imgsz, conf=0.01, iou=iou, max_det=max_det, end2end=end2end,
                            augment=augment, half=True, verbose=False)
        for p, r in zip(chunk, res):
            recs.append(dict(img=p, shape=r.orig_shape,
                             xyxy=r.boxes.xyxy.cpu().numpy(), conf=r.boxes.conf.cpu().numpy()))
    pickle.dump(recs, open(out, 'wb'))
    return recs

def prepare(recs, size_ref=640):
    """Ghép nhãn với dự đoán từng ảnh, gside là cạnh ngắn của nhãn quy về 640."""
    per = []
    for r in recs:
        gt = gt_xyxy(r['img'], r['shape'])
        tp, gconf = greedy_match(gt, r['xyxy'], r['conf'])
        s = size_ref / max(r['shape'][:2])
        gside = np.minimum(gt[:, 2] - gt[:, 0], gt[:, 3] - gt[:, 1]) * s if len(gt) else np.zeros(0)
        per.append(dict(img=r['img'], gt=gt, pr=r['xyxy'], conf=r['conf'], tp=tp, gconf=gconf, gside=gside))
    return per

CONF_GRID = np.round(np.arange(0.02, 0.61, 0.01), 2)

def sweep(per, grid=CONF_GRID):
    n_gt = np.array([len(p['gt']) for p in per]); rows = []
    for t in grid:
        cnt = np.array([(p['conf'] >= t).sum() for p in per])
        tp = np.array([(p['tp'] & (p['conf'] >= t)).sum() for p in per])
        e = cnt - n_gt; R = tp.sum() / max(n_gt.sum(), 1); P = tp.sum() / max(cnt.sum(), 1)
        rows.append(dict(conf=t, MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()), bias=e.mean(),
                         recall=R, precision=P, F1=2 * P * R / (P + R + 1e-9),
                         sev_acc=(severity(cnt) == severity(n_gt)).mean(), max_pred=int(cnt.max())))
    return pd.DataFrame(rows)

def best_row(df):
    return df.loc[df.MAE.idxmin()]

def per_image(per, t):
    n_gt = np.array([len(p['gt']) for p in per]); cnt = np.array([(p['conf'] >= t).sum() for p in per])
    tp = np.array([(p['tp'] & (p['conf'] >= t)).sum() for p in per])
    return pd.DataFrame(dict(img=[Path(p['img']).name for p in per], n_gt=n_gt, n_pred=cnt,
                             TP=tp, FP=cnt - tp, FN=n_gt - tp, err=cnt - n_gt))

SIZE_EDGES = [8, 16, 32]; SIZE_NAMES = ['<8px', '8-16px', '16-32px', '>32px']   # px quy về 640
def recall_by_size(per, t):
    side = np.concatenate([p['gside'] for p in per]); hit = np.concatenate([p['gconf'] >= t for p in per])
    b = np.digitize(side, SIZE_EDGES)
    return pd.DataFrame([dict(size=SIZE_NAMES[i], n_gt=int((b == i).sum()),
                              recall=hit[b == i].mean() if (b == i).any() else np.nan) for i in range(4)])

# Cấu hình train, lý do xem trong báo cáo
BASE_ARGS = dict(
    data=DATA_YAML, seed=SEED, deterministic=True, project=str(RUNS), exist_ok=True,
    # cố định AdamW vì 'auto' sẽ đổi sang MuSGD khi quá 10k iteration
    optimizer='AdamW', lr0=0.002, momentum=0.9, weight_decay=0.0005,
    lrf=0.01, cos_lr=True,
    warmup_epochs=3, warmup_bias_lr=0.0,
    hsv_h=0.005,  # màu đỏ là dấu hiệu viêm nên ít xoay hue thôi
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,  # mun đã nhỏ sẵn, đừng thu thêm
    shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.0, copy_paste=0.0,
    max_det=500,  # có ảnh 434 mụn, mặc định 300 ko đủ
    cache='ram', workers=2, amp=True, plots=True,
)

# Train có resume và tự dừng khi sắp hết giờ
class PauseTraining(Exception): pass

def retarget_ckpt(ckpt_path, project, data_yaml):
    """Sửa đường dẫn trong checkpoint để resume được khi đổi giữa Colab và Kaggle."""
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    a = ck.get('train_args', {})
    new_dir = str(Path(project) / Path(ckpt_path).parent.parent.name)
    if a.get('data') == str(data_yaml) and a.get('save_dir') == new_dir: return
    a.update(data=str(data_yaml), project=str(project), save_dir=new_dir)
    ck['train_args'] = a; torch.save(ck, ckpt_path)
    print('Đã sửa đường dẫn trong', ckpt_path)

def hours_left():
    return SESSION_HOURS - (time.time() - SESSION_T0) / 3600

def run_train(name, weights='yolo26m.pt', stop_epoch=None, **overrides):
    """Train có resume, tự dừng khi hết SESSION_HOURS. Xong trả save_dir, tạm dừng trả None."""
    save_dir = RUNS / name; done = save_dir / 'DONE.json'; last = save_dir / 'weights' / 'last.pt'
    if done.exists():
        print(f'{name} đã xong, bỏ qua.'); return save_dir
    if hours_left() < 0.25:
        print(f'Phiên chỉ còn {hours_left():.2f} giờ, chưa bắt đầu {name}. Phiên sau chạy lại cell.'); return None
    args = {**BASE_ARGS, **overrides, 'name': name}
    def pause_cb(tr):  # chạy sau khi last.pt đã lưu
        ep = tr.epoch + 1
        if ep >= tr.epochs: return  # epoch cuối thì để ultralytics tự kết thúc
        if hours_left() < 0.15 or (stop_epoch and ep >= stop_epoch):
            raise PauseTraining(f'{name}: dừng sau epoch {ep}/{tr.epochs}. last.pt đã lưu, phiên sau chạy lại cell để train tiếp.')
    try:
        if last.exists():
            retarget_ckpt(last, RUNS, DATA_YAML)
            model = YOLO(str(last)); model.add_callback('on_fit_epoch_end', pause_cb)
            print(f'Resume {name} (phiên còn {hours_left():.1f} giờ)')
            try:
                model.train(resume=True)
            except PauseTraining:
                raise
            except Exception as e:
                msg = str(e).lower()
                if 'finished' in msg or 'nothing to resume' in msg: print('Run đã kết thúc trước đó.')
                else: raise
        else:
            model = YOLO(weights); model.add_callback('on_fit_epoch_end', pause_cb)
            print(f'Bắt đầu {name} (phiên còn {hours_left():.1f} giờ)')
            model.train(**args)
    except PauseTraining as e:
        print(e); return None
    json.dump(dict(finished=time.strftime('%Y-%m-%d %H:%M'), args=args), open(done, 'w'), indent=1, default=str)
    return save_dir

def pack_run(name, out_dir):
    """Nén run thành <out_dir>/<name>_run.zip."""
    z = shutil.make_archive(str(Path(out_dir) / f'{name}_run'), 'zip', root_dir=RUNS, base_dir=name)
    print('Đã tạo', z, f'({Path(z).stat().st_size / 1e6:.0f} MB)'); return z

print(f'Đã nạp hàm tiện ích | IMGSZ={IMGSZ} | RUN_NAME={RUN_NAME} | phiên còn {hours_left():.1f} giờ')

## 0. Thống kê dữ liệu
Chạy lại nếu muốn xem tỉ lệ khung dưới 8 px ở 640, 960 và 1280.


In [ ]:
# Cell 0: Thống kê dữ liệu
rows = []
for split in ['train', 'val', 'test']:
    for img in list_images(split):
        w, h = exif_size(Image.open(img))
        lab = read_labels(img)
        n = -1 if lab is None else len(lab)
        sides = np.minimum(lab[:, 3] * w, lab[:, 4] * h) if (lab is not None and len(lab)) else np.zeros(0)
        rows.append(dict(split=split, img=img.name, w=w, h=h, n=n, sides=sides))
df_stat = pd.DataFrame(rows)

for split, g in df_stat.groupby('split', sort=False):
    s640 = np.concatenate([sd * 640 / max(a, b) for sd, a, b in zip(g.sides, g.w, g.h)])
    n = g.n.clip(lower=0)
    print(f"\n== {split}: {len(g)} ảnh | thiếu nhãn {(g.n < 0).sum()} | nhãn rỗng {(g.n == 0).sum()} | "
          f"mụn: tổng {n.sum()}, TB {n.mean():.1f}, trung vị {n.median():.0f}, max {n.max()}")
    print('   cạnh ngắn khung (px) @640  p5/p25/p50/p75/p95 =', np.percentile(s640, [5, 25, 50, 75, 95]).round(1))
    print('   tỉ lệ khung < 8 px theo imgsz:', {sz: f'{(s640 * sz / 640 < 8).mean():.0%}' for sz in (640, 960, 1280)})
    print('   trung vị cạnh ngắn theo imgsz:', {sz: round(float(np.median(s640)) * sz / 640, 1) for sz in (640, 960, 1280)})

## 1. Kiểm tra nhãn và rò rỉ dữ liệu
- 1a: lỗi định dạng nhãn.
- 1b: rò rỉ, tức ảnh trùng hoặc gần trùng giữa train, val, test (md5, perceptual hash, tên gốc Roboflow `xxx_jpg.rf.<hash>`). Có rò rỉ thì val và test sẽ đẹp hơn thực tế.
- ACNE04 có thể có nhiều ảnh của cùng một người, nhưng không có ID bệnh nhân nên kiểu rò rỉ này không kiểm tra được.

Kết quả kiểm tra trên bộ dữ liệu hiện tại nằm ở mục 2 của báo cáo.


In [ ]:
# Cell 1a: Kiểm tra định dạng nhãn
issues = []
for split in ['train', 'val', 'test']:
    for img in list_images(split):
        lp = label_path(img)
        if not lp.exists(): issues.append((split, img.name, 'thiếu file nhãn')); continue
        boxes = []
        for ln in lp.read_text().strip().splitlines():
            v = ln.split()
            if len(v) != 5: issues.append((split, img.name, f'dòng {len(v)} cột (polygon?)')); continue
            c, xc, yc, bw, bh = map(float, v)
            if c != 0: issues.append((split, img.name, f'class {c:g}'))
            if bw <= 0 or bh <= 0: issues.append((split, img.name, 'w/h <= 0'))
            if min(xc - bw / 2, yc - bh / 2) < -0.01 or max(xc + bw / 2, yc + bh / 2) > 1.01:
                issues.append((split, img.name, 'khung tràn ngoài ảnh'))
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
        if len(boxes) > 1:
            iou = box_iou(np.array(boxes), np.array(boxes))
            nd = int((np.triu(iou, 1) > 0.7).sum())
            if nd: issues.append((split, img.name, f'{nd} cặp khung trùng (IoU>0.7)'))
df_iss = pd.DataFrame(issues, columns=['split', 'img', 'van_de'])
df_iss.to_csv(EVAL / 'label_issues.csv', index=False)
print('Tổng số vấn đề:', len(df_iss))
print(df_iss.groupby(['split', 'van_de']).size() if len(df_iss) else 'Không có lỗi định dạng.')

In [ ]:
# Cell 1b: kiểm tra rò rỉ train/val/test
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imagehash'])
import imagehash

meta, hashes, md5s = [], [], []
for split in ['train', 'val', 'test']:
    for img in list_images(split):
        im = Image.open(img).convert('L'); im.thumbnail((256, 256))
        hashes.append(imagehash.phash(im).hash.flatten())
        md5s.append(hashlib.md5(open(img, 'rb').read()).hexdigest())
        meta.append((split, img.name))
sp = np.array([m[0] for m in meta])

# trùng y hệt
cross_md5 = pd.DataFrame(dict(split=sp, md5=md5s)).groupby('md5').split.nunique()
print('Ảnh trùng md5 giữa các split:', int((cross_md5 > 1).sum()))

# gần trùng, phash Hamming <= 6
H = np.array(hashes, dtype=bool)
dist = (H[:, None, :] != H[None, :, :]).sum(2)
iu = np.triu_indices(len(meta), 1)
mask = (dist[iu] <= 6) & (sp[iu[0]] != sp[iu[1]])
pairs = pd.DataFrame([(meta[i][0], meta[i][1], meta[j][0], meta[j][1], int(dist[i, j]))
                      for i, j in zip(iu[0][mask], iu[1][mask])], columns=['split_a', 'img_a', 'split_b', 'img_b', 'hamming'])
pairs.to_csv(EVAL / 'near_duplicates_cross_split.csv', index=False)
print('Cặp gần trùng giữa các split:', len(pairs))
if len(pairs): print(pairs.groupby(['split_a', 'split_b']).size())

# cùng ảnh gốc roboflow (augment cuả 1 ảnh rơi vào 2 split)
src = pd.Series([n.split('.rf.')[0] for _, n in meta])
if src.str.len().mean() < pd.Series([n for _, n in meta]).str.len().mean():
    cross_src = pd.DataFrame(dict(split=sp, src=src)).groupby('src').split.nunique()
    print('Ảnh gốc Roboflow xuất hiện ở >1 split:', int((cross_src > 1).sum()))
else:
    print('Tên file không theo mẫu Roboflow (.rf.), bỏ qua.')

## 2. Train dài yolo26m ở 1280
Lý do chọn tham số nằm ở mục 5 của báo cáo. `batch=4` là vừa đủ VRAM của T4 ở 1280, OOM thì giảm còn 2. `save_period=20` để giữ checkpoint đo overfit ở mục 3.

Cell tự dừng khi phiên còn khoảng 10 phút, lúc đó last.pt đã lưu. Phiên sau chạy S1 đến S3 rồi chạy lại cell này.
Muốn dừng ở epoch cụ thể thì thêm `stop_epoch=50`, lần sau nhớ bỏ đi.


In [ ]:
# Cell 2: train dài, mỗi phiên chạy lại cell này đến khi báo đã xong
TRAIN = dict(imgsz=IMGSZ, batch=4, epochs=100, patience=30, close_mosaic=15, save_period=20)
run_train(RUN_NAME, **TRAIN)

## 3. Overfit và độ ổn định
Loss train tính trên ảnh có augment còn loss val thì không, nên hai số này không so được. Lúc screening `overfit_gap` ra số âm cũng vì vậy.

Ở đây chạy val trên chính tập train, không augment, rồi lấy `gap = mAP_train - mAP_val`. Gap tăng mà val vẫn tốt lên thì bình thường, val đi ngang hoặc xấu đi thì là overfit.

Độ ổn định thì xem std ở 10 epoch cuối, muốn chắc hơn thì chạy thêm seed ở Cell 3c.


In [ ]:
# Cell 3a: Đường cong huấn luyện
rdir = RUNS / RUN_NAME
r = pd.read_csv(rdir / 'results.csv'); r.columns = r.columns.str.strip()
targs = yaml.safe_load(open(rdir / 'args.yaml'))
close_ep = targs['epochs'] - targs['close_mosaic']

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for c in [c for c in r.columns if c.endswith('loss')]:
    ax[0].plot(r.epoch, r[c], '--' if c.startswith('train') else '-', label=c)
ax[0].set_title('Loss (train có augment, val thì không, đừng so trực tiếp)')
for c in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']: ax[1].plot(r.epoch, r[c], label=c)
for c in ['metrics/precision(B)', 'metrics/recall(B)']: ax[2].plot(r.epoch, r[c], label=c)
for a in ax:
    if close_ep <= r.epoch.max(): a.axvline(close_ep, color='gray', ls=':', label='tắt mosaic')
    a.legend(fontsize=7); a.grid(alpha=.3)
plt.show()

last10 = r.tail(10)
best_ep = int(r.loc[r['metrics/mAP50-95(B)'].idxmax(), 'epoch'])
print(f"Số epoch đã chạy: {len(r)} | best epoch (mAP50-95): {best_ep}")
print(f"10 epoch cuối: mAP50-95 = {last10['metrics/mAP50-95(B)'].mean():.4f} ± {last10['metrics/mAP50-95(B)'].std():.4f} | "
      f"recall = {last10['metrics/recall(B)'].mean():.4f} ± {last10['metrics/recall(B)'].std():.4f}")

In [ ]:
# Cell 3b: đánh giá các checkpoint, có cache nên ngắt giữa chừng ko sao
EVAL_ON_TRAIN = True  # 963 ảnh train chạy khá lâu, ko cần gap thì để False
wdir = rdir / 'weights'
ckpts = sorted(wdir.glob('epoch*.pt'), key=lambda p: int(p.stem.replace('epoch', ''))) + [wdir / 'best.pt', wdir / 'last.pt']
cache_csv = EVAL / f'{RUN_NAME}_ckpt_eval.csv'
ck_df = pd.read_csv(cache_csv) if cache_csv.exists() else pd.DataFrame(columns=['ckpt'])
kw = dict(data=DATA_YAML, imgsz=IMGSZ, batch=8, half=True, max_det=500, plots=False, verbose=False)
for w in ckpts:
    if w.name in set(ck_df.ckpt): continue
    b = best_row(sweep(prepare(collect_preds(w, 'val'))))
    mv = YOLO(str(w)).val(split='val', **kw)
    row = dict(ckpt=w.name, val_mAP5095=mv.box.map, val_MAE=b.MAE, val_recall=b.recall, best_conf=b.conf)
    if EVAL_ON_TRAIN:
        mt = YOLO(str(w)).val(split='train', **kw)
        row.update(train_mAP5095_noaug=mt.box.map, gap=mt.box.map - mv.box.map)
    ck_df = pd.concat([ck_df, pd.DataFrame([row])], ignore_index=True); ck_df.to_csv(cache_csv, index=False)
display(ck_df.round(4))

In [ ]:
# Cell 3c: thêm 2 seed để đo độ ổn định, tốn như 2 run dài nên mặc định tắt
RUN_EXTRA_SEEDS = False
if RUN_EXTRA_SEEDS:
    stats = []
    for s in [SEED, 0, 1]:
        name = RUN_NAME if s == SEED else f'{RUN_NAME}_s{s}'
        sd = run_train(name, **{**TRAIN, 'seed': s, 'save_period': -1 if s != SEED else TRAIN['save_period']})
        if sd is None: break  # hết giờ thif chạy lại sau
        b = best_row(sweep(prepare(collect_preds(RUNS / name / 'weights' / 'best.pt', 'val'))))
        stats.append(dict(seed=s, MAE=b.MAE, recall=b.recall, conf=b.conf))
    if stats:
        st = pd.DataFrame(stats); display(st)
        print(f"MAE = {st.MAE.mean():.2f} ± {st.MAE.std():.2f} | recall = {st.recall.mean():.3f} ± {st.recall.std():.3f}")

## 4. Quét hậu xử lý trên val
YOLO26 mặc định không dùng NMS (end2end) nên `iou` không có tác dụng, `max_det` chỉ là trần số hộp. `end2end=False` mới dùng head one-to-many với NMS, lúc đó `iou` mới có ý nghĩa.

Quét cả hai rồi chọn MAE thấp nhất, hoà thì lấy recall cao hơn. Khi model đếm thiếu thì ngưỡng tốt nhất thường thấp hơn 0.25.


In [ ]:
# Cell 4: quét và chốt cấu hình
FINAL_WEIGHTS = RUNS / RUN_NAME / 'weights' / 'best.pt'  # đổi sang epochXX.pt nếu cell 3b tốt hơn
configs = [dict(end2end=True, iou=0.7, max_det=500)] + [dict(end2end=False, iou=i, max_det=500) for i in (0.4, 0.5, 0.6, 0.7)]
tab, curves = [], {}
for c in configs:
    df = sweep(prepare(collect_preds(FINAL_WEIGHTS, 'val', **c)))
    b = best_row(df); tab.append({**c, **b.to_dict()}); curves[f"e2e={c['end2end']},iou={c['iou']}"] = df
tab = pd.DataFrame(tab).sort_values(['MAE', 'recall'], ascending=[True, False])
display(tab.round(4))
ch = tab.iloc[0]
if ch.max_pred >= ch.max_det: print('Số hộp chạm max_det, phải tăng max_det')

CHOICE = dict(weights=str(FINAL_WEIGHTS), imgsz=IMGSZ, conf=float(ch.conf), end2end=bool(ch.end2end), iou=float(ch.iou),
              max_det=int(ch.max_det), augment=False, val_MAE=float(ch.MAE), val_recall=float(ch.recall),
              val_bias=float(ch.bias), sev_bins=SEV_BINS, sev_names=SEV_NAMES, sev_ascii=SEV_ASCII)
json.dump(CHOICE, open(EVAL / 'postproc_choice.json', 'w'), indent=1, ensure_ascii=False)
print('Đã chốt:', CHOICE)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, df in curves.items(): ax[0].plot(df.conf, df.MAE, label=k); ax[1].plot(df.conf, df.recall, label=k)
ax[0].set(title='MAE vs conf (val)'); ax[1].set(title='Recall vs conf (val)')
for a in ax: a.legend(fontsize=7); a.grid(alpha=.3)
plt.show()

In [ ]:
# Cell 4b: thử TTA trên val, ko chắc chạy đươc với head end2end nên để trong try
# TTA chậm hơn 2-3 lần, MAE giảm rõ thì mới sửa CHOICE rồi lưu lại json
try:
    df_tta = sweep(prepare(collect_preds(FINAL_WEIGHTS, 'val', end2end=CHOICE['end2end'],
                                         iou=CHOICE['iou'], max_det=CHOICE['max_det'], augment=True)))
    b = best_row(df_tta)
    print(f"TTA: conf={b.conf:.2f} MAE={b.MAE:.2f} recall={b.recall:.3f}  (không TTA: MAE={CHOICE['val_MAE']:.2f})")
except Exception as e:
    print('TTA không chạy được với cấu hình này:', e)

## 5. Đánh giá test
Cell có khoá, đã có `test_result.json` thì chỉ in lại.

Test có nhiều mụn nhỏ hơn val (60% so với 47% khung dưới 8 px ở 640).


In [ ]:
# Cell 5: Test
TEST_FILE = EVAL / 'test_result.json'
if TEST_FILE.exists():
    print('Test đã chạy rồi, chỉ in lại:'); print(json.dumps(json.load(open(TEST_FILE)), indent=1, ensure_ascii=False))
else:
    C = json.load(open(EVAL / 'postproc_choice.json'))
    per_t = prepare(collect_preds(C['weights'], 'test', C['imgsz'], end2end=C['end2end'], iou=C['iou'],
                                  max_det=C['max_det'], augment=C['augment']))
    s = sweep(per_t, grid=[C['conf']]).iloc[0]
    pi_t = per_image(per_t, C['conf'])
    m = YOLO(C['weights']).val(data=DATA_YAML, split='test', imgsz=C['imgsz'], batch=8, half=True,
                               end2end=C['end2end'], max_det=C['max_det'], plots=False, verbose=False)
    out = dict(conf=C['conf'], n_img=len(pi_t), MAE=float(s.MAE), RMSE=float(s.RMSE), bias=float(s.bias),
               recall=float(s.recall), precision=float(s.precision), sev_acc=float(s.sev_acc),
               mAP50=float(m.box.map50), mAP50_95=float(m.box.map), val_MAE=C['val_MAE'], val_recall=C['val_recall'])
    json.dump(out, open(TEST_FILE, 'w'), indent=1); pi_t.to_csv(EVAL / 'test_per_image.csv', index=False)
    print(json.dumps(out, indent=1))
    names_map = dict(enumerate(SEV_NAMES))
    print('\nMa trận mức độ (hàng = nhãn, cột = dự đoán):')
    print(pd.crosstab(pd.Series(severity(pi_t.n_gt)).map(names_map).rename('nhãn'),
                      pd.Series(severity(pi_t.n_pred)).map(names_map).rename('dự đoán')))

## 6. Phân tích lỗi trên val
Màu: xanh lá là mụn tìm thấy, vàng đậm là bỏ sót (FN), đỏ là báo nhầm (FP, có ghi conf).

Mục này để xem lỗi nằm ở nhãn hay ở model. FP conf cao mà trông như mụn thật thì là thiếu nhãn, FP có IoU 0.1 đến 0.5 thì chỉ lệch khung, còn FN dồn vào nhóm dưới 8 px thì vẫn thiếu độ phân giải.


In [ ]:
# Cell 6a: bảng lỗi theo ảnh, theo kích thước, theo mức độ
import cv2
C = json.load(open(EVAL / 'postproc_choice.json')); T = C['conf']
per_v = prepare(collect_preds(C['weights'], 'val', C['imgsz'], end2end=C['end2end'], iou=C['iou'], max_det=C['max_det']))
pi_v = per_image(per_v, T); pi_v.to_csv(EVAL / 'val_per_image.csv', index=False)
print('Top 10 ảnh sai đếm nhiều nhất:'); display(pi_v.reindex(pi_v.err.abs().sort_values(ascending=False).index).head(10))
print('\nRecall theo kích thước (px quy về 640):'); display(recall_by_size(per_v, T).round(3))
pi_v['muc_do'] = pd.Series(severity(pi_v.n_gt)).map(dict(enumerate(SEV_NAMES)))
print('\nSai số theo mức độ (theo nhãn):')
display(pi_v.groupby('muc_do').agg(so_anh=('err', 'size'), MAE=('err', lambda e: e.abs().mean()), bias=('err', 'mean')).round(2))

In [ ]:
# Cell 6b: Vẽ 6 ảnh tệ nhất
def draw(p, t):
    im = cv2.imread(p['img']); lw = max(2, int(max(im.shape[:2]) / 500))
    for g, gc in zip(p['gt'].astype(int), p['gconf']):
        ok = gc >= t
        cv2.rectangle(im, tuple(g[:2]), tuple(g[2:]), (0, 200, 0) if ok else (0, 255, 255), lw if ok else 2 * lw)
    keep = (p['conf'] >= t) & ~p['tp']
    for b, c in zip(p['pr'][keep].astype(int), p['conf'][keep]):
        cv2.rectangle(im, tuple(b[:2]), tuple(b[2:]), (0, 0, 255), lw)
        cv2.putText(im, f'{c:.2f}', (b[0], max(0, b[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, lw / 3, (0, 0, 255), max(1, lw // 2))
    return im[:, :, ::-1]

viz_dir = EVAL / 'viz_worst'; viz_dir.mkdir(exist_ok=True)
worst = pi_v.err.abs().sort_values(ascending=False).index[:6]
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for a, i in zip(axes.flat, worst):
    im = draw(per_v[i], T); cv2.imwrite(str(viz_dir / pi_v.img[i]), im[:, :, ::-1])
    a.imshow(im); a.axis('off')
    a.set_title(f"{pi_v.img[i][:30]}  GT={pi_v.n_gt[i]} pred={pi_v.n_pred[i]} FN={pi_v.FN[i]} FP={pi_v.FP[i]}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 6c: xuất crop FP conf cao và FN để soát nhãn bằng tay (lưu trên Drive)
rev = EVAL / 'review'; (rev / 'fp_highconf').mkdir(parents=True, exist_ok=True); (rev / 'fn').mkdir(exist_ok=True)
def crop(im, b, k=1.5):
    x1, y1, x2, y2 = b.astype(int); pad = int(max(x2 - x1, y2 - y1) * k) + 10
    ox, oy = max(0, x1 - pad), max(0, y1 - pad)
    c = im[oy:y2 + pad, ox:x2 + pad].copy()
    cv2.rectangle(c, (x1 - ox, y1 - oy), (x2 - ox, y2 - oy), (0, 0, 255), 1)
    return c
fp_rows, fn_n = [], 0
for p in per_v:
    m_fp = (~p['tp']) & (p['conf'] >= 0.5); m_fn = p['gconf'] < T
    if not (m_fp.any() or m_fn.any()): continue
    im = cv2.imread(p['img']); stem = Path(p['img']).stem[:40]
    for k, (b, c) in enumerate(zip(p['pr'][m_fp], p['conf'][m_fp])):
        io = box_iou(b[None], p['gt']).max() if len(p['gt']) else 0.0
        cv2.imwrite(str(rev / 'fp_highconf' / f'{stem}_{k}_c{c:.2f}_iou{io:.2f}.jpg'), crop(im, b))
        fp_rows.append(dict(img=stem, conf=c, iou_max_gt=io))
    for k, g in enumerate(p['gt'][m_fn]):
        cv2.imwrite(str(rev / 'fn' / f'{stem}_{k}.jpg'), crop(im, g)); fn_n += 1
fp = pd.DataFrame(fp_rows, columns=['img', 'conf', 'iou_max_gt'])
print(f"FP conf≥0.5: {len(fp)} | trong đó IoU 0.1–0.5 với GT (lệch khung): {((fp.iou_max_gt > 0.1) & (fp.iou_max_gt < 0.5)).sum()}")
print(f"FN (bỏ sót) ở conf={T:.2f}: {fn_n}")
print('Crop lưu ở', rev)

## 7. SAHI (suy luận theo ô)
Chỉ thử nếu mục 6 cho thấy FN vẫn dồn vào nhóm mụn nhỏ.

Model train trên ảnh nguyên thu về 1280, nên khi cắt ô mụn sẽ to hơn lúc train. `scale=0.3` chỉ phủ khoảng 0.7 đến 1.3 lần, vì vậy cạnh ô nên bằng cạnh dài ảnh chia 1.3. Muốn SAHI thật sự có lợi thì thường phải train trên ô, việc này khá lớn nên để sau.

Chưa thử SAHI với YOLO26 end2end ở bản 8.4.192, nên cell để trong try.


In [ ]:
# Cell 7: SAHI trên val
RUN_SAHI = False
if RUN_SAHI:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sahi'])
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        C = json.load(open(EVAL / 'postproc_choice.json'))
        det = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=C['weights'], confidence_threshold=0.01,
                                                 device='cuda:0', image_size=C['imgsz'])
        recs_s = []
        for img in list_images('val'):
            W, Hh = exif_size(Image.open(img)); SL = int(max(W, Hh) / 1.3)
            r = get_sliced_prediction(str(img), det, slice_height=SL, slice_width=SL, overlap_height_ratio=0.2,
                                      overlap_width_ratio=0.2, perform_standard_pred=True, postprocess_type='NMS',
                                      postprocess_match_threshold=0.5, verbose=0)
            ops = r.object_prediction_list
            recs_s.append(dict(img=str(img), shape=(r.image.size[1], r.image.size[0]),
                               xyxy=np.array([o.bbox.to_xyxy() for o in ops]).reshape(-1, 4),
                               conf=np.array([o.score.value for o in ops])))
        b = best_row(sweep(prepare(recs_s)))
        print(f"SAHI: conf={b.conf:.2f} MAE={b.MAE:.2f} recall={b.recall:.3f}  (gốc: MAE={C['val_MAE']:.2f})")
    except Exception as e:
        print('SAHI lỗi:', e)

## 8. Đóng gói cho máy RTX 3050
Thư mục `deploy/` trên Drive gồm `best.pt`, `deploy_config.json` và `count_acne.py`.
Trên máy 3050 cài đúng `ultralytics==8.4.192` và torch có CUDA, chép thư mục `deploy/` về rồi chạy:
```
python count_acne.py --source duong_dan_anh_hoac_thu_muc --save
```
- `half=True` tự bật khi có GPU, `imgsz` lấy từ config nên đúng bằng 1280 lúc train.
- Muốn nhanh hơn có thể export TensorRT ngay trên máy 3050: `YOLO('best.pt').export(format='engine', half=True, imgsz=1280)`.
- Mức độ chỉ dựa trên số mụn thấy trong ảnh, để tham khảo, không thay cho chẩn đoán.


In [ ]:
# Cell 8: tạo thư mục deploy
C = json.load(open(EVAL / 'postproc_choice.json'))
DEPLOY = DRIVE_ROOT / 'deploy'; DEPLOY.mkdir(exist_ok=True)
shutil.copy(C['weights'], DEPLOY / 'best.pt')
json.dump({k: C[k] for k in ['imgsz', 'conf', 'end2end', 'iou', 'max_det', 'augment', 'sev_bins', 'sev_names', 'sev_ascii']},
          open(DEPLOY / 'deploy_config.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False)

SCRIPT = r'''"""Đếm mụn và xếp mức độ. Chạy: python count_acne.py --source <ảnh hoặc thư mục> [--save] [--out ket_qua]
Yêu cầu: ultralytics==8.4.192 (cùng phiên bản lúc train), torch có CUDA."""
import argparse, csv, json
from pathlib import Path
import cv2, torch
from ultralytics import YOLO

HERE = Path(__file__).resolve().parent
CFG = json.load(open(HERE / 'deploy_config.json', encoding='utf-8'))

def muc_do(n):
    for i, b in enumerate(CFG['sev_bins']):
        if n <= b:
            return i
    return len(CFG['sev_bins'])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--source', required=True)
    ap.add_argument('--save', action='store_true', help='lưu ảnh có vẽ khung')
    ap.add_argument('--out', default='ket_qua')
    a = ap.parse_args()
    cuda = torch.cuda.is_available()
    model = YOLO(str(HERE / 'best.pt'))
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    rows = []
    for r in model.predict(a.source, imgsz=CFG['imgsz'], conf=CFG['conf'], iou=CFG['iou'], max_det=CFG['max_det'],
                           end2end=CFG['end2end'], augment=CFG['augment'], half=cuda, device=0 if cuda else 'cpu',
                           stream=True, verbose=False):
        n = len(r.boxes); k = muc_do(n); name = Path(r.path).name
        rows.append(dict(anh=name, so_mun=n, muc_do=CFG['sev_names'][k]))
        print(f'{name}: {n} mụn, mức {CFG["sev_names"][k]}')
        if a.save:
            im = r.plot(labels=False, conf=False, line_width=2)
            cv2.putText(im, f'{n} mun - {CFG["sev_ascii"][k]}', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 5)
            cv2.putText(im, f'{n} mun - {CFG["sev_ascii"][k]}', (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 2)
            cv2.imwrite(str(out / name), im)
    with open(out / 'dem_mun.csv', 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=['anh', 'so_mun', 'muc_do']); w.writeheader(); w.writerows(rows)
    print('Đã lưu', out / 'dem_mun.csv')

if __name__ == '__main__':
    main()
'''
(DEPLOY / 'count_acne.py').write_text(SCRIPT, encoding='utf-8')
print('Đã tạo', DEPLOY, [p.name for p in DEPLOY.iterdir()])

# chạy thử trên colab vs 1 ảnh test
sample = str(list_images('test')[0])
subprocess.run([sys.executable, str(DEPLOY / 'count_acne.py'), '--source', sample, '--out', '/content/demo_out'])

## 9. Chuyển run giữa Colab và Kaggle
Kaggle có quota GPU riêng, mỗi phiên tối đa 12 tiếng. Chọn GPU T4 cho giống Colab, đổi GPU thì kết quả không tái lập chính xác.

Từ Colab sang Kaggle:
1. Chạy Cell 9a để có `<run>_run.zip` trên Drive rồi tải về máy.
2. Trên Kaggle tạo một Dataset từ thư mục dữ liệu (nén .zip, Kaggle không tự giải nén rar) và một Dataset từ `<run>_run.zip`.
3. Tạo notebook mới, chọn GPU T4, bật Internet, Add Input cả hai Dataset.
4. Dùng `acne_kaggle_resume.ipynb`, hoặc chạy Cell 9K thay cho S1 và S2 rồi chạy S3 và Cell 2. Nên chạy bằng Save Version, Save & Run All để chạy nền.
5. Xong thì chạy Cell 9b và tải `<run>_run.zip` ở tab Output.

Từ Kaggle về Colab: upload zip lên Drive, chạy Cell 9c để giải nén vào `runs/`, rồi chạy lại Cell 2. Cell này tự sửa đường dẫn và resume.


In [ ]:
# Cell 9a (Colab): nén run lên Drive để mang đi
pack_run(RUN_NAME, '/content/drive/MyDrive')

In [ ]:
# Cell 9K (Kaggle): thay cho S1 và S2, sau đó chạy S3 và Cell 2 như bình thường
import os, sys, json, time, pickle, hashlib, subprocess, shutil, importlib.util
from pathlib import Path
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.192'], check=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, ultralytics, yaml
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '|', torch.cuda.get_device_name(0))

DRIVE_ROOT = Path('/kaggle/working/acne_yolo26')
RUNS = DRIVE_ROOT / 'runs'; EVAL = DRIVE_ROOT / 'eval'
for p in (RUNS, EVAL): p.mkdir(parents=True, exist_ok=True)
IMGSZ = 1280; RUN_NAME = f'y26m_{IMGSZ}'; SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_DIR = {'train': 'train', 'val': 'valid', 'test': 'test'}
SESSION_T0 = time.time(); SESSION_HOURS = 11.0  # kaggle cắt ở 12h

def find_data_root(base):
    for p in [base, *sorted(base.rglob('*'))]:
        if p.is_dir() and (p / 'train' / 'images').is_dir() and (p / 'valid' / 'images').is_dir():
            return p

DATA_ROOT = find_data_root(Path('/kaggle/input'))
assert DATA_ROOT, 'Chưa Add Input dataset acne04v2_yolo'
DATA_YAML = '/kaggle/working/acne_data_fixed.yaml'
orig_yaml = list(DATA_ROOT.glob('*.yaml'))
names = yaml.safe_load(open(orig_yaml[0])).get('names', ['acne']) if orig_yaml else ['acne']
yaml.safe_dump(dict(path=str(DATA_ROOT), train='train/images', val='valid/images', test='test/images', names=names),
               open(DATA_YAML, 'w'), allow_unicode=True)
print('DATA_ROOT =', DATA_ROOT)

# chép run từ colab sang chỗ ghi đươc, /kaggle/input chỉ đọc
for last in Path('/kaggle/input').rglob('weights/last.pt'):
    dst = RUNS / last.parent.parent.name
    if not dst.exists():
        shutil.copytree(last.parent.parent, dst); print('Đã chép run:', dst.name)

In [ ]:
# Cell 9b (Kaggle): nén run để tải về ở tab Output
pack_run(RUN_NAME, '/kaggle/working')

In [ ]:
# Cell 9c (Colab): giải nén run mang từ Kaggle về, zip đã upload lên Drive
ZIP_ON_DRIVE = f'/content/drive/MyDrive/{RUN_NAME}_run.zip'
shutil.unpack_archive(ZIP_ON_DRIVE, RUNS)
print('Đã giải nén vào', RUNS, sorted(p.name for p in RUNS.iterdir()))